# FluxAudio-S 正式續訓（formal v2）

這份 notebook 是唯一的正式訓練入口，預設從 **60,000** 續訓到 **70,000**。

重要原則：

- Checkpoint、資料壓縮檔與權重永久保存在 Google Drive。
- `/content` 是 Colab 暫存空間；每次換執行階段都會重新建立專案與環境。
- 不在 notebook 內保存任何 Hugging Face token。
- 不執行舊版的 FAD/KL 評估，因為舊流程曾把生成音訊與自己的 cache 比較，結果無效。
- 訓練完成後先做固定 prompt／seed 比較，再決定是否進入下一階段。

建議訓練使用 **L4 GPU**。開始前請確認本次真的有足夠時間與運算單元。


## 1. 本次正式設定

通常只需要修改這一格。預設不會直接啟動訓練。


In [ ]:
from pathlib import Path

# 本次階段：60k -> 70k
SOURCE_ITERATION = 60_000
TARGET_ITERATION = 70_000
EXP_ID = "fluxaudio_s_70k_stage3"

BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 8
LEARNING_RATE = 3e-5

# 安全開關：完成所有檢查後，才手動改成 True
CONFIRM_TRAINING = False

REPO = Path("/content/ICME26-ATTM-GC-FluxAudio")
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_ZIP = DRIVE_ROOT / "FluxAudio_data/jamendo_meanaudio_ready_cache.zip"
WEIGHT_CACHE = DRIVE_ROOT / "FluxAudio_data/meanaudio_weights"

SOURCE_CHECKPOINT = (
    DRIVE_ROOT
    / "FluxAudio_checkpoints/fluxaudio_s_60k_stage2/fluxaudio_s_60k_stage2_ckpt_last.pth"
)
OUTPUT_DIR = DRIVE_ROOT / "FluxAudio_checkpoints" / EXP_ID
EMA_FOLDER = DRIVE_ROOT / "FluxAudio_checkpoints/fluxaudio_s_50k/ema_ckpts"

print(f"來源：{SOURCE_ITERATION:,}")
print(f"目標：{TARGET_ITERATION:,}")
print("實驗名稱：", EXP_ID)
print("訓練確認：", CONFIRM_TRAINING)


## 2. GPU 與 Google Drive

這格會先處理可能擋住掛載的 Colab 本機假資料夾，且不會刪除其中內容。


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import time

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu.returncode != 0:
    raise RuntimeError("未偵測到 GPU。請先將 Colab 執行階段改成 L4 GPU。")
print("GPU：", gpu.stdout.strip())

mountpoint = Path("/content/drive")
expected_drive = mountpoint / "MyDrive/FluxAudio_checkpoints"

if not expected_drive.exists():
    if mountpoint.exists() and any(mountpoint.iterdir()):
        backup = Path(f"/content/drive_local_backup_{int(time.time())}")
        mountpoint.rename(backup)
        print("未掛載時建立的本機資料已移到：", backup)
    mountpoint.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mountpoint))
else:
    print("Drive 已掛載")

if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Google Drive 掛載失敗")

print("Drive 檢查完成")


## 3. 檢查 Drive 永久檔案

在建立任何輸出資料夾前，先確認來源 checkpoint、資料與五個必要權重都存在。


In [ ]:
required_weights = [
    "v1-16.pth",
    "best_netG.pt",
    "empty_string_t5.pth",
    "empty_string_clap_c.pth",
    "music_speech_audioset_epoch_15_esc_89.98.pt",
]

required_paths = [SOURCE_CHECKPOINT, DATA_ZIP, EMA_FOLDER]
required_paths += [WEIGHT_CACHE / name for name in required_weights]

missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Drive 缺少必要檔案：\n" + "\n".join(str(path) for path in missing)
    )

print("來源 checkpoint：", SOURCE_CHECKPOINT)
print("資料壓縮檔：", round(DATA_ZIP.stat().st_size / 1024**3, 2), "GB")
print("必要權重：", len(required_weights), "個，全部存在")
print("EMA 資料夾：存在")


## 4. 建立乾淨專案與安裝套件

GitHub 專案位於 `/content`，因此每次更換 Colab 執行階段都需要重建；這不是重新訓練。


In [ ]:
import shutil
import subprocess
import sys
import time

if REPO.exists() and not (REPO / "pyproject.toml").exists():
    backup = Path(f"/content/FluxAudio_incomplete_{int(time.time())}")
    REPO.rename(backup)
    print("不完整專案已移到：", backup)

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/ntu-musicailab/ICME26-ATTM-GC-FluxAudio.git",
            str(REPO),
        ],
        check=True,
    )
else:
    print("專案已存在，不重複下載")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)],
    check=True,
)

print("專案：", REPO)
print("infer.py：", (REPO / "infer.py").exists())
print("train.py：", (REPO / "train.py").exists())


## 5. 複製必要權重

權重從 Drive cache 複製到本次 Colab 暫存專案，不會再次從網路下載。


In [ ]:
import shutil

target_weights = REPO / "weights"
target_weights.mkdir(parents=True, exist_ok=True)

for name in required_weights:
    source = WEIGHT_CACHE / name
    destination = target_weights / name
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        print("複製：", name)
        shutil.copy2(source, destination)
    else:
        print("已存在：", name)

print("權重準備完成")


## 6. 套用兩個必要相容修正

1. 關閉訓練不需要的 `av_bench` 強制匯入。  
2. 允許訓練完成後跳過耗時的自動大量取樣。


In [ ]:
from pathlib import Path
import re

def patch_optional_av_bench(path: Path):
    text = path.read_text(encoding="utf-8")
    if "AV_BENCH_OPTIONAL_PATCH" in text:
        print(path.name, "：已修正")
        return

    pattern = re.compile(
        r"^from av_bench\.evaluate import evaluate\s*\n"
        r"from av_bench\.extract import extract\s*$",
        re.MULTILINE,
    )
    replacement = (
        "# AV_BENCH_OPTIONAL_PATCH\n"
        "try:\n"
        "    from av_bench.evaluate import evaluate\n"
        "    from av_bench.extract import extract\n"
        "except Exception:\n"
        "    evaluate = None\n"
        "    extract = None"
    )
    updated, count = pattern.subn(replacement, text, count=1)
    if count != 1:
        raise RuntimeError(f"無法在 {path} 找到 av_bench 匯入位置")
    path.write_text(updated, encoding="utf-8")
    print(path.name, "：av_bench 已改為非必要")


for runner_name in ["runner_flowmatching.py", "runner_meanflow.py"]:
    patch_optional_av_bench(REPO / "meanaudio" / runner_name)


train_path = REPO / "train.py"
train_text = train_path.read_text(encoding="utf-8")

if "SKIP_FINAL_SAMPLE_PATCH" not in train_text:
    sample_calls = list(re.finditer(r"(?m)^([ \t]*)sample\(", train_text))
    if not sample_calls:
        raise RuntimeError("找不到 train.py 最後的 sample()；停止以避免錯誤修改")
    match = sample_calls[-1]
    indent = match.group(1)
    guard = (
        f"{indent}# SKIP_FINAL_SAMPLE_PATCH\n"
        f"{indent}if cfg.get('skip_final_sample', False):\n"
        f"{indent}    print('Skipping final EMA synthesis and test sampling.')\n"
        f"{indent}    return\n\n"
    )
    train_text = train_text[:match.start()] + guard + train_text[match.start():]
    train_path.write_text(train_text, encoding="utf-8")
    print("train.py：skip-final 修正完成")
else:
    print("train.py：skip-final 已修正")

print("必要修正完成")


## 7. 還原訓練 NPZ 資料

只有正式訓練需要這一步。若三個 split 數量正確，就不會重複解壓。預期約需 15–25 分鐘與約 58 GB 空間。


In [ ]:
import shutil
import subprocess

data_root = REPO / "data/jamendo_meanaudio_ready"
expected_counts = {"train": 166496, "val": 299, "test": 300}

def npz_count(split):
    folder = data_root / split / "npz"
    return len(list(folder.glob("*.npz"))) if folder.exists() else 0

counts_before = {split: npz_count(split) for split in expected_counts}
print("目前 NPZ：", counts_before)

if counts_before != expected_counts:
    free_gb = shutil.disk_usage("/content").free / 1024**3
    print("可用空間：", round(free_gb, 1), "GB")
    if free_gb < 75:
        raise RuntimeError("本機磁碟不足；建議至少保留 75 GB")

    print("開始從 Drive 解壓訓練資料……")
    subprocess.run(
        ["unzip", "-q", "-o", str(DATA_ZIP), "-d", str(REPO)],
        check=True,
    )

counts_after = {split: npz_count(split) for split in expected_counts}
print("還原後 NPZ：", counts_after)

if counts_after != expected_counts:
    raise RuntimeError(
        f"NPZ 數量不正確。預期 {expected_counts}，實際 {counts_after}"
    )

print("訓練資料準備完成")


## 8. 建立輸出連結與啟動前總檢查

訓練輸出直接寫入 Drive。這格不會啟動訓練。


In [ ]:
import os

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
local_exp = REPO / "exps" / EXP_ID
local_exp.parent.mkdir(parents=True, exist_ok=True)

if local_exp.is_symlink():
    if local_exp.resolve() != OUTPUT_DIR.resolve():
        raise RuntimeError(f"既有輸出連結指向其他位置：{local_exp.resolve()}")
elif local_exp.exists():
    raise RuntimeError(f"本機已有非連結輸出資料夾：{local_exp}")
else:
    os.symlink(OUTPUT_DIR, local_exp, target_is_directory=True)

checks = {
    "GPU": subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0,
    "來源 checkpoint": SOURCE_CHECKPOINT.exists(),
    "訓練資料": all(npz_count(s) == n for s, n in expected_counts.items()),
    "五個權重": all((REPO / "weights" / n).exists() for n in required_weights),
    "輸出連結": local_exp.is_symlink(),
}

for name, ok in checks.items():
    print(f"{name}：", "OK" if ok else "失敗")

if not all(checks.values()):
    raise RuntimeError("啟動前檢查失敗，請勿開始訓練")

print("\n輸出位置：", OUTPUT_DIR)
print("來源：", SOURCE_CHECKPOINT)
print("下一格仍有 CONFIRM_TRAINING 安全開關")


## 9. 正式續訓：60k → 70k

只有確定要開始扣 GPU 運算單元時，才回到第一格把：

```python
CONFIRM_TRAINING = True
```

然後重新執行第一格與本格。訓練過程請保持 Colab 連線。Checkpoint 每 1,000 steps 保存一次到 Drive。


In [ ]:
import os
import subprocess

if not CONFIRM_TRAINING:
    raise RuntimeError(
        "尚未確認訓練。若確定要開始，請把第一格 CONFIRM_TRAINING 改成 True。"
    )

command = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=1",
    "train.py",
    "--config-name", "train_config_jamendo.yaml",
    f"exp_id={EXP_ID}",
    f"checkpoint={SOURCE_CHECKPOINT}",
    "model=fluxaudio_s",
    f"batch_size={BATCH_SIZE}",
    f"eval_batch_size={EVAL_BATCH_SIZE}",
    f"num_iterations={TARGET_ITERATION}",
    "text_encoder_name=t5_clap",
    "data_dim.text_c_dim=512",
    "++use_rope=True",
    "use_meanflow=False",
    "cfg_strength=4.5",
    f"learning_rate={LEARNING_RATE}",
    "linear_warmup_steps=100",
    "lr_schedule=constant",
    "++reset_scheduler_on_load=False",
    "val_interval=1000",
    "eval_interval=999999",
    "save_eval_interval=999999",
    "log_extra_interval=999999",
    "log_text_interval=50",
    "save_weights_interval=1000",
    "save_checkpoint_interval=1000",
    "++skip_final_sample=True",
    "++use_wandb=False",
    "pin_memory=False",
    f"num_workers={NUM_WORKERS}",
    "ac_oversample_rate=5",
    "compile=False",
    f"ema.checkpoint_folder={EMA_FOLDER}",
]

environment = os.environ.copy()
environment["OMP_NUM_THREADS"] = "1"

print(f"開始正式續訓：{SOURCE_ITERATION:,} → {TARGET_ITERATION:,}")
print("輸出：", OUTPUT_DIR)

result = subprocess.run(
    command,
    cwd=REPO,
    env=environment,
)

print("Return code：", result.returncode)
if result.returncode != 0:
    raise RuntimeError("訓練失敗；請保留畫面中的第一段真正 traceback")

print("正式續訓完成")


## 10. 驗證70k輸出

確認完整 checkpoint 與純模型權重都已寫入 Drive，才能關閉執行階段。


In [ ]:
checkpoint_70k = OUTPUT_DIR / f"{EXP_ID}_ckpt_last.pth"
weights_70k = OUTPUT_DIR / f"{EXP_ID}_last.pth"

for path in [checkpoint_70k, weights_70k]:
    exists = path.exists()
    size = path.stat().st_size / 1024**3 if exists else 0
    print(path.name, "| 存在：", exists, "| 大小：", round(size, 2), "GB")

if not checkpoint_70k.exists() or checkpoint_70k.stat().st_size < 2 * 1024**3:
    raise RuntimeError("70k 完整 checkpoint 尚未安全保存")
if not weights_70k.exists() or weights_70k.stat().st_size < 400 * 1024**2:
    raise RuntimeError("70k 模型權重尚未安全保存")

print("70k 已安全保存至 Drive，可以關閉 GPU 執行階段")


## 11. 固定條件比較（可選）

這一段只在70k訓練完成後執行。使用同樣的 prompt、seed與推論參數比較60k和70k。Rock用來判斷節奏與樂器結構；Electronic用來判斷節拍與高頻；Ambient僅作輔助。


In [ ]:
from IPython.display import Audio, display
import shutil
import subprocess
import sys
import time

models = [
    (
        "60k",
        DRIVE_ROOT / "FluxAudio_checkpoints/fluxaudio_s_60k_stage2/fluxaudio_s_60k_stage2_last.pth",
    ),
    ("70k", OUTPUT_DIR / f"{EXP_ID}_last.pth"),
]

tests = [
    (
        "rock_seed42",
        42,
        "Energetic alternative rock music with distorted electric guitars, "
        "punchy bass, powerful acoustic drums, a memorable melody, clear song "
        "structure, and polished modern production.",
    ),
    (
        "electronic_seed42",
        42,
        "Upbeat electronic dance music with a clear four-on-the-floor kick, "
        "crisp hi-hats, bright synthesizer melody, rhythmic bass line, strong "
        "build-up and energetic modern production.",
    ),
]

comparison_dir = DRIVE_ROOT / "FluxAudio_comparisons/60k_vs_70k"
comparison_dir.mkdir(parents=True, exist_ok=True)
temp_root = Path(f"/content/fluxaudio_compare_{int(time.time())}")
temp_root.mkdir(parents=True, exist_ok=True)

for model_name, checkpoint in models:
    if not checkpoint.exists():
        raise FileNotFoundError(checkpoint)

final_files = []
for test_name, seed, prompt in tests:
    for model_name, checkpoint in models:
        tag = f"{test_name}_{model_name}"
        temp_dir = temp_root / tag
        temp_dir.mkdir(parents=True, exist_ok=True)

        command = [
            sys.executable, "infer.py",
            "--variant", "fluxaudio_s",
            "--model_path", str(checkpoint),
            "--encoder_name", "t5_clap",
            "--use_rope",
            "--text_c_dim", "512",
            "--prompt", prompt,
            "--duration", "9.975",
            "--cfg_strength", "4.5",
            "--num_steps", "25",
            "--seed", str(seed),
            "--output", str(temp_dir),
        ]

        print("開始生成：", tag)
        result = subprocess.run(command, cwd=REPO)
        if result.returncode != 0:
            raise RuntimeError(f"{tag} 生成失敗")

        generated = list(temp_dir.rglob("*.wav")) + list(temp_dir.rglob("*.flac"))
        if not generated:
            raise FileNotFoundError(f"{tag} 找不到生成音訊")

        source = max(generated, key=lambda p: p.stat().st_mtime)
        destination = comparison_dir / f"{tag}{source.suffix}"
        shutil.copy2(source, destination)
        final_files.append(destination)
        print("完成：", destination)

print("\n保存位置：", comparison_dir)
for path in final_files:
    print("\n", path.name)
    display(Audio(filename=str(path)))


## 12. 結束本次 Colab

確認上一格顯示70k checkpoint 與模型權重都存在後：

1. 儲存 notebook。
2. 選擇 **執行階段 → 中斷連線並刪除執行階段**。
3. 不要讓 GPU 在無工作時保持連線，否則仍會扣運算單元。

下一階段是否前往80k，應由60k／70k固定條件比較結果決定，不直接盲跑到200k。
